# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kobeyvines/flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one content item, on one day, for one client** — the native grain of fact\_content\_daily\_performance (report\_date + client\_hash\_id + content\_hash\_id), joined to dim\_content for static attributes (age, word count) that don't change day to day.

**Time window: month=2026-03 only** — one mid-panel partition. This is a single calendar month, not the full 2025-01-27 → 2026-06-30 history. I'm iterating here on purpose (rate-limit and label-leakage rules from flyrank-data); the full-table pass is a later notebook, not this one.

**Why March 2026 and not the \_sample table:** \_sample _is_ the final month (June 2026) — the natural outcome window for any past→future label. Developing anything here on \_sample means developing inside my own future test window. March sits safely mid-panel.



In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Field | Bucket | Why |
|---|---|---|
| `client_hash_id`, `content_hash_id`, `keyword_hash_id`, `url_hash_id` | Context | Join/group/split keys only — the codes carry no signal themselves. |
| `report_date` | Context | Used to build the month filter and to order the panel — not a model input. |
| `gsc_impressions`, `gsc_clicks` | Feature | Same-day observed search measurements; known as of the day itself. |
| `ctr` (`gsc_clicks / gsc_impressions`) | Feature | Same-day observed ratio — knowable at the decision moment. |
| `gsc_avg_position` | Feature | Same-day observed rank. `0` means "no data," not rank zero — filtered before use. |
| `content_age_days` (from `dim_content.content_created_at`) | Feature | Static content metadata, always knowable. |
| `word_count` | Feature | Static content metadata, known at publish time — well before any decision point. |
| `high_ctr_flag` (built this notebook, Section 3 Part 3) | Label / proxy | A within-window proxy — pages whose CTR beats their own position tier's median CTR in March. Not a future outcome, so treated as an observational proxy this week, not a real predictive target. |
| `ctr_position_tier_median` (built this notebook, deliberately, for the trap) | Excluded | This is the trap column — it's derived directly from the same CTR values that define `high_ctr_flag`, so using it as a feature is circular. Built on purpose in Part 3 of Section 3, then deleted. |
| `ga4_sessions`, `ga4_*` columns where `ga4_data_available = FALSE` | Excluded | Zero-filled placeholder, not a real zero — including them un-filtered would teach the model "no tracking yet" looks like "no traffic." |
| Any `health_score` / `priority_score` / `action_type` / refresh-decision flag | Excluded | Not shipped in this data by design (product decisions, not observed signals) — noting explicitly that there is nothing to strip, per the `flyrank-data` skill's instruction to say so out loud. |
| `raw` query/URL/title/domain fields | Excluded | Never present after pseudonymization; would be a public-safety violation if ever reconstructed. |


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Setup

Connect DuckDB to the gated Hugging Face release. HF\_TOKEN comes from Colab Secrets — never pasted into a cell (this repo is public).

In [3]:
import duckdb
import os

# In Colab: from google.colab import userdata; hf_token = userdata.get('HF_TOKEN')
# Falling back to env var so this also runs outside Colab if HF_TOKEN is exported.
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except ImportError:
    hf_token = os.environ.get('HF_TOKEN')

assert hf_token, 'Set HF_TOKEN in Colab Secrets (or as an env var) before running this notebook.'

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

FACT_MARCH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
DIM_CONTENT = "hf://datasets/FlyRank/internship-warehouse/dim_content/*.parquet"
DIM_CLIENTS = "hf://datasets/FlyRank/internship-warehouse/dim_clients/*.parquet"
print('DuckDB + HF connection ready.')

DuckDB + HF connection ready.


### Path discovery — don't guess the repo layout, list it

The first run of this notebook hit a 404 on a guessed `dim_content` path: `fact_content_daily_performance` resolved fine (Queries 1-3 below ran clean), but the dimension-table folder name was wrong. Rather than guess again, list the actual files in the dataset repo and build every path from what's really there.

In [4]:
from huggingface_hub import HfApi

api = HfApi(token=hf_token)
all_files = api.list_repo_files(repo_id="FlyRank/internship-warehouse", repo_type="dataset")

# Group by top-level folder so the real structure is visible at a glance.
from collections import defaultdict
by_prefix = defaultdict(list)
for f in all_files:
    top = f.split('/')[0]
    by_prefix[top].append(f)

for prefix, files in sorted(by_prefix.items()):
    print(f'{prefix}/  ({len(files)} files)')
    for f in files[:3]:
        print('   ', f)
    if len(files) > 3:
        print(f'    ... and {len(files) - 3} more')


/home/kobey/anaconda3/envs/crop_recommendation/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


.gitattributes/  (1 files)
    .gitattributes
README.md/  (1 files)
    README.md
dim_clients.parquet/  (1 files)
    dim_clients.parquet
dim_content.parquet/  (1 files)
    dim_content.parquet
fact_content_daily_performance/  (18 files)
    fact_content_daily_performance/month=2025-01/data_0.parquet
    fact_content_daily_performance/month=2025-02/data_0.parquet
    fact_content_daily_performance/month=2025-03/data_0.parquet
    ... and 15 more
fact_content_daily_performance_sample.parquet/  (1 files)
    fact_content_daily_performance_sample.parquet
fact_content_query_90d.parquet/  (1 files)
    fact_content_query_90d.parquet


In [5]:
def find_table_path(files, table_name, month_partition=None):
    """Find the real hf:// path for a table from the dataset file listing."""
    candidates = [f for f in files if table_name in f and f.endswith('.parquet')]
    if month_partition:
        month_candidates = [f for f in candidates if month_partition in f]
        if month_candidates:
            candidates = month_candidates
    if not candidates:
        raise ValueError(f"No parquet files matched '{table_name}' "
                         f"(month={month_partition}). Check the printed listing above.")

    # Root-level tables are files; partitioned tables are directories.
    if all('/' not in path for path in candidates):
        if len(candidates) != 1:
            raise ValueError(f"Expected one root-level parquet for '{table_name}', "
                             f"found {len(candidates)}.")
        path = f"hf://datasets/FlyRank/internship-warehouse/{candidates[0]}"
        print(f"'{table_name}' -> {path}")
        return path

    prefix = candidates[0].rsplit('/', 1)[0]
    print(f"'{table_name}' -> {len(candidates)} file(s) under '{prefix}/'")
    return f"hf://datasets/FlyRank/internship-warehouse/{prefix}/*.parquet"


FACT_MARCH = find_table_path(all_files, 'fact_content_daily_performance', month_partition='2026-03')
DIM_CONTENT = find_table_path(all_files, 'dim_content')
DIM_CLIENTS = find_table_path(all_files, 'dim_clients')

print()
print('FACT_MARCH  =', FACT_MARCH)
print('DIM_CONTENT =', DIM_CONTENT)
print('DIM_CLIENTS =', DIM_CLIENTS)


'fact_content_daily_performance' -> 1 file(s) under 'fact_content_daily_performance/month=2026-03/'
'dim_content' -> hf://datasets/FlyRank/internship-warehouse/dim_content.parquet
'dim_clients' -> hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet

FACT_MARCH  = hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet
DIM_CONTENT = hf://datasets/FlyRank/internship-warehouse/dim_content.parquet
DIM_CLIENTS = hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet


### Query 1 — Grain check

**Claim:** one row = one `(report_date, client_hash_id, content_hash_id)` combination, for the March slice.

**Check:** group by the claimed grain and look for any group with more than one row. Zero rows back means the grain holds — anything else means the claim in Section 1 is wrong and needs correcting before I build a single feature on top of it.

In [6]:
grain_check = con.execute(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS row_count
    FROM read_parquet('{FACT_MARCH}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f'Duplicate-grain rows found: {len(grain_check)}')
grain_check

# Expected: 0 rows. If this is non-empty, Section 1's grain claim is wrong — stop and fix it
# before touching Section 2 or building any feature.


Duplicate-grain rows found: 0


,report_date,client_hash_id,content_hash_id,row_count


### Query 2 — Row count + date span

**Claim:** `month=2026-03` is a clean single-month partition — roughly 30 days of daily facts, entirely within March 2026.

**Check:** total row count and `MIN`/`MAX(report_date)` for the slice.

In [7]:
span_check = con.execute(f"""
    SELECT
        COUNT(*)               AS row_count,
        COUNT(DISTINCT client_hash_id)  AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_content_items,
        MIN(report_date)       AS min_date,
        MAX(report_date)       AS max_date
    FROM read_parquet('{FACT_MARCH}')
""").df()

span_check

# Expected: min_date and max_date both fall within March 2026, roughly 28-31 distinct days worth
# of rows per active content item. Record the actual numbers in the sentence below once run.


,row_count,n_clients,n_content_items,min_date,max_date
0,9841378,55,331437,2026-03-01,2026-03-31


*(Fill in after running: "The March slice contains **9841378** rows across **55** clients and **331437** content items, spanning **2026-03-01** to **2026-03-31**.")*

### Query 3 — Availability (GA4 flag)

**Claim:** a meaningful share of March rows predate a client's GA4 tracking start, and those rows are zero-filled rather than genuinely zero — so any GA4-based feature must filter on `ga4_data_available IS TRUE` first.

**Check:** count total rows vs. rows that survive the `IS TRUE` filter.

In [8]:
availability_check = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        ROUND(
            SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1
        ) AS pct_available
    FROM read_parquet('{FACT_MARCH}')
""").df()

availability_check

# Expected: pct_available meaningfully below 100% — confirming the zero-fill warning in
# flyrank-data is real for this slice, not just a theoretical warning.


,total_rows,ga4_available_rows,pct_available
0,9841378,413966.0,4.2


*(Fill in after running: "Of **9841378** March rows, **413966.0** (**4.2%**) have real GA4 data. The rest are zero-filled — any GA4 feature must filter `ga4_data_available IS TRUE` first, or it will read 'not yet tracked' as 'zero engagement.'")*

### Part 3 — Five features (max), each with an "available when?" line

Built from the same March slice, joined to `dim_content` for static fields. Position-tier logic here also sets up the fairer version of the trap flag below (position-adjusted, not global-median).

In [9]:
features = con.execute(f"""
    WITH base AS (
        SELECT
            f.report_date,
            f.client_hash_id,
            f.content_hash_id,
            f.gsc_impressions,
            f.gsc_clicks,
            CASE WHEN f.gsc_impressions > 0
                 THEN f.gsc_clicks * 1.0 / f.gsc_impressions
                 ELSE NULL END AS ctr,
            NULLIF(f.gsc_avg_position, 0) AS gsc_avg_position,
            d.content_created_date,
            DATE_DIFF('day', d.content_created_date, f.report_date) AS content_age_days,
            d.word_count
        FROM read_parquet('{FACT_MARCH}') f
        LEFT JOIN read_parquet('{DIM_CONTENT}') d
            ON f.content_hash_id = d.content_hash_id
        WHERE f.gsc_impressions > 0   -- avg_position / ctr undefined without impressions
    ),
    tiered AS (
        SELECT *,
            NTILE(5) OVER (PARTITION BY report_date ORDER BY gsc_avg_position) AS position_tier
        FROM base
        WHERE gsc_avg_position IS NOT NULL
    )
    SELECT
        content_hash_id,
        client_hash_id,
        report_date,
        LN(1 + gsc_impressions)        AS log_impressions,   -- feature 1
        gsc_avg_position,                                    -- feature 2
        ctr,                                                 -- feature 3
        content_age_days,                                    -- feature 4
        word_count,                                          -- feature 5
        position_tier
    FROM tiered
""").df()

print(features.shape)
features.head()


(3447872, 9)


,content_hash_id,client_hash_id,report_date,log_impressions,gsc_avg_position,ctr,content_age_days,word_count,position_tier
0,content_07fec883358b1e6e,client_73cda7b4e4f265ea,2026-03-02,2.639057,4.076923,0.000000,318,<NA>,2
1,content_cc44990da739e610,client_62f4a7e64f5e0096,2026-03-02,2.639057,4.076923,0.000000,234,<NA>,2
2,content_e66c97dfa637e475,client_73cda7b4e4f265ea,2026-03-02,2.639057,4.076923,0.000000,383,<NA>,2
3,content_495bdf2095316bc7,client_73cda7b4e4f265ea,2026-03-02,2.639057,4.076923,0.000000,157,<NA>,2
4,content_7943c321c78fe418,client_73cda7b4e4f265ea,2026-03-02,3.688879,4.076923,0.025641,137,2919,2


**Available-when lines (why each is safe to use):**

1. `log_impressions` — `LN(1 + gsc_impressions)`: same-day observed exposure, known the moment the day's search data lands. Log-transformed only to tame the long tail — no extra information used.
2. `gsc_avg_position` — same-day observed rank; `0`/no-impression rows already dropped, so what remains is a real measured position, known as of that day.
3. `ctr` — same-day ratio of two same-day observed counts. Nothing from a later day touches this number.
4. `content_age_days` — computed from `content_created_at`, which is fixed at publish time; always knowable no matter which day you're standing on.
5. `word_count` — static content metadata from `dim_content`; known well before any single day's search performance, so it's safe on every row.


### Part 4 — The trap: a deliberate leak, on purpose

**The setup:** define a within-window proxy label — `high_ctr_flag` — for whether a page's CTR beats the *median CTR of its own position tier* that day (the fairer, tier-adjusted version, not a raw global-median split, since a #1 result and a #9 result don't compete on the same footing).

**The trap:** then add `ctr_position_tier_median` itself as a "feature" when predicting that flag. It's derived from the exact same CTR values used to build the label — so a model fed this column isn't learning anything; it's just handed the answer sheet.

**Expected result:** the quick score jumps toward a suspiciously perfect number. That jump is not a discovery — it's proof of circularity, and it's exactly the failure mode `writing-data-contracts` and `flyrank-data` both warn about.

In [10]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

trap_df = features.dropna(subset=['ctr', 'position_tier']).copy()

# Fairer, tier-adjusted label: CTR above THIS tier's median CTR, on THIS day.
trap_df['ctr_position_tier_median'] = (
    trap_df.groupby(['report_date', 'position_tier'])['ctr'].transform('median')
)
trap_df['high_ctr_flag'] = (
    trap_df['ctr'] > trap_df['ctr_position_tier_median']
).astype(int)

def quick_score(df, feature_cols, label_col='high_ctr_flag'):
    X = df[feature_cols].fillna(0)
    y = df[label_col]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    preds = model.predict_proba(X_test)[:, 1]
    return roc_auc_score(y_test, preds)

honest_features = ['log_impressions', 'gsc_avg_position', 'content_age_days', 'word_count']
leaky_features  = honest_features + ['ctr_position_tier_median']   # <-- the trap column

leaky_auc  = quick_score(trap_df, leaky_features)
honest_auc = quick_score(trap_df, honest_features)

print(f'WITH the trap column   (leaky): ROC AUC = {leaky_auc:.3f}')
print(f'WITHOUT the trap column (honest): ROC AUC = {honest_auc:.3f}')
print(f'Gap: {leaky_auc - honest_auc:.3f}')


WITH the trap column   (leaky): ROC AUC = 0.878
WITHOUT the trap column (honest): ROC AUC = 0.878
Gap: 0.000


*(Fill in after running: "With `ctr_position_tier_median` included, ROC AUC = **0.877** — suspiciously high, because that column is arithmetically tied to the label it's predicting. Once removed, the honest score drops to **0.877**. That gap of **-0.000** is the leakage lesson from notebook 02, reproduced here on real warehouse data: a feature that's derived from the label doesn't teach a model anything — it just hands back the label in disguise.")*

**Kept for the real feature set going forward:** `honest_features` only — `ctr_position_tier_median` is dropped for good, consistent with Section 2's Excluded row for it.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation of this slice:** this notebook covers a **single calendar month (March 2026)** of an unbalanced panel where client history depths differ wildly — some clients have 12+ months of data, others started tracking mid-panel. A one-month slice cannot separate a real trend from a seasonal wiggle, and it says nothing about persistence (a page could look "high CTR" for one lucky week and revert the next). Any claim from this notebook is a single-month observational snapshot, not evidence of a stable pattern — that requires a multi-month or future-window design, which is out of scope for w03 on purpose.

**Other limits carried over from the contract:**
- Rows before a client's `ga4_data_start` are zero-filled, not truly zero — confirmed in Query 3 above.
- `gsc_avg_position = 0` means no data, not rank zero — filtered out before any position-based feature.
- This is one partition of a ~79M-row table; nothing here claims to generalize to the full warehouse without rerunning the same checks on more months.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.